<a href="https://colab.research.google.com/github/Rachani02/Statistical-Learning-e22282/blob/main/Assignment_7c_Item_Response_Prediction_and_Click_Through_Rate_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Q. Bayesian Estimation of a User Ability Parameter from Item Responses

1. Visualizing the Mechanics of the 2PL Model

The Item Characteristic Curve (ICC) under the 2PL model is given by:

$$p_i(\theta) = \frac{1}{1 + e^{-a_i(\theta - b_i)}}$$


In [1]:
import numpy as np
import plotly.graph_objects as go

# Define theta grid
theta = np.linspace(-4, 4, 400)

def p_i(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

# Create Plotly figure
fig = go.Figure()

# Varying b_i with fixed a_i = 1.5
fig.add_trace(go.Scatter(x=theta, y=p_i(theta, a=1.5, b=-1.0), mode='lines', name='a=1.5, b=-1.0 (Easy)'))
fig.add_trace(go.Scatter(x=theta, y=p_i(theta, a=1.5, b=0.0), mode='lines', name='a=1.5, b=0.0 (Medium)'))
fig.add_trace(go.Scatter(x=theta, y=p_i(theta, a=1.5, b=1.0), mode='lines', name='a=1.5, b=1.0 (Hard)'))

# Additional slope comparison (a_i = 0.5 vs 2.0 at b = 0)
fig.add_trace(go.Scatter(x=theta, y=p_i(theta, a=0.5, b=0.0), mode='lines', line=dict(dash='dash'), name='a=0.5, b=0.0 (Low Discrim)'))
fig.add_trace(go.Scatter(x=theta, y=p_i(theta, a=2.0, b=0.0), mode='lines', line=dict(dash='dot'), name='a=2.0, b=0.0 (High Discrim)'))

fig.update_layout(
    title="2PL Item Characteristic Curves (ICCs)",
    xaxis_title="Latent Ability (θ)",
    yaxis_title="P(Y = 1 | θ)",
    template="plotly_white"
)
fig.show()

Interpretation of Difficulty ($b_i$)

 *The difficulty parameter $b_i$ represents the point on the ability axis $\theta$ where $p_i(b_i) = 0.5$ (a $50\%$ probability of answering correctly).

 *Horizontal Shift: Increasing $b_i$ shifts the entire sigmoid curve horizontally to the right. This implies that to maintain a $50\%$ success probability on a harder item, the user must possess a higher latent ability level $\theta$.

 ---


2. Sequential Likelihood Contribution

For a single item $k$, the likelihood contribution given response $y_k \in \{0, 1\}$ is Bernoulli-distributed:

$$L(y_k \mid \theta) = [p_k(\theta)]^{y_k} [1 - p_k(\theta)]^{1 - y_k} = \frac{\exp\left(y_k \cdot a_k(\theta - b_k)\right)}{1 + \exp\left(a_k(\theta - b_k)\right)}$$

Assuming response independence conditional on $\theta$, the joint likelihood function for the running history vector $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ is the product of individual item likelihoods:

$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^k [p_i(\theta)]^{y_i} [1 - p_i(\theta)]^{1 - y_i} = \prod_{i=1}^k \frac{\exp\left(y_i \cdot a_i(\theta - b_i)\right)}{1 + \exp\left(a_i(\theta - b_i)\right)}$$

---


3. Mathematical Formulation of the Running Update

By Bayes' Theorem, the posterior density at step $k$ given history $\mathbf{y}^{(k)}$ is updated recursively using the posterior density from step $k-1$ as the current prior:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) = \frac{L(y_k \mid \theta) \, f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})}{\int_{-\infty}^{\infty} L(y_k \mid \theta') \, f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta' \mid \mathbf{y}^{(k-1)}) \, d\theta'}$$

Expressed up to a constant of proportionality:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto L(y_k \mid \theta) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$

where $f_{\Theta \mid \mathbf{Y}^{(0)}}(\theta) = f_{\Theta}^{(0)}(\theta) = \frac{1}{\sqrt{2\pi}} e^{-\theta^2 / 2}$.

---


4. Dynamic Shifting Analysis

When a user correctly answers ($y_k = 1$) an item with high difficulty ($b_k \gg 0$):

1. The single-item likelihood function becomes $L(y_k = 1 \mid \theta) = p_k
(\theta) = \frac{1}{1 + e^{-a_k(\theta - b_k)}}$.

2. This is a monotonically increasing function of $\theta$ that transitions sharply around $b_k$.

3. Multiplying the prior density $f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta)$ by this monotonically increasing function severely suppresses the probability density for $\theta < b_k$ while preserving or amplifying the density for $\theta > b_k$.

4. Result: The peak (MAP) and expected value (Posterior Mean) of the resulting distribution shift significantly to the right (towards higher ability values), providing strong positive evidence of the user's proficiency.

---

5. Tracking Certainty and Sharpness

The discrimination parameter $a_k$ determines the maximum slope of $p_k(\theta)$ at $\theta = b_k$, which directly impacts Fisher's Information $I(\theta) = a_k^2 p_k(\theta)(1 - p_k(\theta))$:



*   High Discrimination ($a_k \gg 1$): The likelihood $L(y_k \mid \theta)$ acts nearly as a step function. Multiplying the prior by a steep function provides high local information around $b_k$, rapidly cutting off improbable $\theta$ values and narrowing the posterior distribution (reducing posterior variance / increasing sharpness).

*   Low Discrimination ($a_k \approx 0$): The likelihood function is nearly flat across the domain of $\theta$. Multiplying by a flat function alters the prior density negligibly, contributing minimal information and leaving the variance/sharpness virtually unchanged.

---




6. Numerical Implementation of a Running Grid

To maintain $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$ computationally without analytical closed forms:

1. Grid Discretization: Define a discrete evaluation grid $\boldsymbol{\theta} = [\theta_1, \theta_2, \dots, \theta_M]$ over a bounded interval (e.g., $[-4, 4]$) with spacing $\Delta \theta = \frac{\theta_M - \theta_1}{M-1}$.

2. Prior Initialization: Initialize probability vector $\mathbf{q}^{(0)} \in \mathbb{R}^M$ where $q_m^{(0)} = f_{\Theta}^{(0)}(\theta_m)$, then normalize such that $\sum_{m=1}^M q_m^{(0)} \Delta \theta = 1$.

3. Sequential Update Step $k$:


*   Compute likelihood vector $\mathbf{L}_k \in \mathbb{R}^M$ with elements $L_{k, m} = p_k(\theta_m)^{y_k}(1 - p_k(\theta_m))^{1 - y_k}$.

*   Compute unnormalized posterior vector: $\mathbf{q}^* = \mathbf{q}^{(k-1)} \odot \mathbf{L}_k$ (element-wise product).


*   Sequential Normalization Step: Compute the marginal likelihood scalar $Z_k$ using trapezoidal numeric integration:

$$Z_k = \sum_{m=1}^M q_m^* \Delta \theta$$

*   Set the normalized posterior grid vector: $\mathbf{q}^{(k)} = \frac{\mathbf{q}^*}{Z_k}$.

---






7. Evaluating Convergence over Timeline ($n = 20$)

Below is the complete simulation script tracking the Posterior Mean ($\widehat{\theta}_{\text{Bayes}}^{(k)}$) and MAP ($\widehat{\theta}_{\text{MAP}}^{(k)}$) over $20$ items against $\theta_{\text{true}} = 0.75$.

In [3]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# =====================================================================
# PART 1: SEQUENTIAL BAYESIAN UPDATE (MANUAL 4-ITEM SIMULATION)
# =====================================================================

# 1. Define a fine grid of latent ability values (theta)
theta = np.linspace(-5, 5, 500)

# 2. Initialize the prior distribution: Standard Normal N(0, 1)
prior = stats.norm.pdf(theta, 0, 1)

# 3. Define the Item Response Function (2PL Model)
def p_i(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

# 4. Define the manual simulated sequence of item encounters
running_items = [
    {"a": 1.0, "b": -1.5, "y": 1},  # Step 1: Got an easy item correct
    {"a": 1.5, "b": 0.5,  "y": 1},  # Step 2: Got a medium-hard item correct
    {"a": 1.2, "b": 1.5,  "y": 0},  # Step 3: Got a very hard item incorrect
    {"a": 2.0, "b": 0.2,  "y": 1}   # Step 4: Got a highly discriminative item correct
]

# Create the first Plotly figure
fig1 = go.Figure()

# Add the initial Prior distribution trace
fig1.add_trace(go.Scatter(
    x=theta,
    y=prior,
    mode='lines',
    name='Initial Prior: N(0,1)',
    line=dict(dash='dash', width=2.5, color='gray')
))

# Run the sequential update loop
current_posterior = prior.copy()

for idx, item in enumerate(running_items):
    a = item["a"]
    b = item["b"]
    y = item["y"]

    # Calculate item response probability across the grid
    prob = p_i(theta, a, b)

    # Calculate the likelihood contribution of this response
    likelihood = (prob ** y) * ((1 - prob) ** (1 - y))

    # Posterior proportional to Prior * Likelihood
    current_posterior = current_posterior * likelihood

    # Numerically normalize the density curve using the trapezoidal rule
    integral = np.trapezoid(current_posterior, theta)
    current_posterior /= integral

    # Format trace labels
    result_text = "Correct" if y == 1 else "Incorrect"
    trace_name = f"Step {idx+1}: After Item {idx+1} ({result_text}, a={a}, b={b})"

    # Add the current posterior step to the plot
    fig1.add_trace(go.Scatter(
        x=theta,
        y=current_posterior,
        mode='lines',
        name=trace_name,
        line=dict(width=2)
    ))

# Customize layout for Figure 1
fig1.update_layout(
    title={
        'text': "Sequential Bayesian Update of User Ability (θ)",
        'y': 0.95, 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top'
    },
    xaxis_title="Latent Ability Parameter (θ)",
    yaxis_title="Probability Density f(θ | y)",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(
        yanchor="top", y=0.95, xanchor="left", x=0.02,
        bgcolor="rgba(255,255,255,0.7)"
    )
)

# Display the first plot
fig1.show()


# =====================================================================
# PART 2: PERFORMANCE TRACKING & CONVERGENCE TIMELINE (20 ITEMS)
# =====================================================================

# Set random seed for reproducibility of the random items/responses
np.random.seed(42)

# 1. Setup Simulation Parameters
theta_true = 0.75
n_items = 20
theta_grid = np.linspace(-5, 5, 1000)  # Finer grid for tracking accuracy

# 2. Generate Random Item Characteristics
a_params = np.random.uniform(0.5, 2.0, size=n_items)
b_params = np.random.normal(0, 1, size=n_items)

# 3. Initialize Tracking Arrays (Step 0 = Initial Prior State)
running_bayes = [0.0]
running_map = [0.0]
steps = list(range(n_items + 1))

# Initialize prior density distribution: N(0, 1)
current_posterior_sim = stats.norm.pdf(theta_grid, 0, 1)

# 4. Run the 20-item Sequential Simulation loop
for k in range(n_items):
    a_k = a_params[k]
    b_k = b_params[k]

    # Compute true response probability at true theta
    prob_true = p_i(theta_true, a_k, b_k)

    # Simulate stochastically generated user response y_k
    y_k = 1 if np.random.uniform(0, 1) < prob_true else 0

    # Calculate likelihood curve across grid
    prob_grid = p_i(theta_grid, a_k, b_k)
    likelihood = (prob_grid ** y_k) * ((1 - prob_grid) ** (1 - y_k))

    # Update and normalize
    current_posterior_sim = current_posterior_sim * likelihood
    integral_sim = np.trapezoid(current_posterior_sim, theta_grid)
    current_posterior_sim /= integral_sim

    # Compute point estimates
    theta_bayes_k = np.trapezoid(theta_grid * current_posterior_sim, theta_grid)
    theta_map_k = theta_grid[np.argmax(current_posterior_sim)]

    # Store estimates
    running_bayes.append(theta_bayes_k)
    running_map.append(theta_map_k)

# 5. Create the second Plotly figure
fig2 = go.Figure()

# Add True Ability Reference Horizontal Line
fig2.add_hline(
    y=theta_true,
    line_dash="dash",
    line_color="red",
    line_width=2,
    annotation_text=f"True Ability (θ = {theta_true})",
    annotation_position="bottom right"
)

# Add Posterior Mean Trace
fig2.add_trace(go.Scatter(
    x=steps, y=running_bayes,
    mode='lines+markers',
    name='Posterior Mean (θ̂_Bayes)',
    line=dict(color='blue', width=2.5),
    marker=dict(size=6)
))

# Add MAP Trace
fig2.add_trace(go.Scatter(
    x=steps, y=running_map,
    mode='lines+markers',
    name='MAP Estimate (θ̂_MAP)',
    line=dict(color='green', width=2),
    marker=dict(size=6, symbol='square')
))

# Customize Layout for Figure 2
fig2.update_layout(
    title={
        'text': "Convergence of Latent Ability Estimators (θ) Over Time",
        'y': 0.93, 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top'
    },
    xaxis_title="Sequence / Item Position (k)",
    yaxis_title="Estimated Ability (θ̂)",
    xaxis=dict(tickmode='linear', tick0=0, dtick=2),
    yaxis=dict(range=[-1, 2]),
    template="plotly_white",
    hovermode="x unified",
    legend=dict(yanchor="top", y=0.15, xanchor="left", x=0.02)
)

# Display the second plot
fig2.show()

Analysis of Estimator Convergence

1. Distance Reduction: As $k$ increases, both $\widehat{\theta}_{\text{Bayes}}^
{(k)}$ and $\widehat{\theta}_{\text{MAP}}^{(k)}$ oscillate around and progressively converge toward $\theta_{\text{true}} = 0.75$.

2. Variance Shrinkage: The accumulation of independent likelihood multipliers shrinks the width (variance) of the posterior density function.

3. Platform Confidence: The shrinking variance narrows the credible intervals around $\widehat{\theta}$. Mathematically, this reflects an accumulation of Fisher Information, indicating that the platform's uncertainty regarding the user's latent ability decreases as more item responses are collected.

---


#Q. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

1. Structural Probability and Properties

The probability density function (PDF) of a $\text{Beta}(\alpha, \beta)$ distribution over $\theta \in [0, 1]$ is defined as:

$$f_{\Theta}(\theta) = \frac{1}{\mathrm{B}(\alpha, \beta)} \theta^{\alpha - 1} (1 - \theta)^{\beta - 1}$$

In [1]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# Theta domain
theta = np.linspace(0.001, 0.999, 500)

# Parameter pairs
params = [
    (1, 1, 'Uninformative: Alpha=1, Beta=1', 'blue'),
    (2, 8, 'Right-Skewed: Alpha=2, Beta=8', 'orange'),
    (8, 2, 'Left-Skewed: Alpha=8, Beta=2', 'green')
]

fig = go.Figure()

for alpha, beta, label, color in params:
    pdf = stats.beta.pdf(theta, alpha, beta)
    fig.add_trace(go.Scatter(
        x=theta, y=pdf,
        mode='lines',
        name=label,
        line=dict(color=color, width=2.5)
    ))

fig.update_layout(
    title="Beta Distribution Densities under Various Parameter Configurations",
    xaxis_title="Latent Click-Through Rate (θ)",
    yaxis_title="Probability Density",
    template="plotly_white",
    hovermode="x unified"
)

fig.show()

Interpretation of $\alpha$ and $\beta$The parameters $\alpha$ and $\beta$ can be interpreted as pseudo-counts of observed successes (clicks) and failures (non-clicks) plus $1$:



*   $\alpha = 1, \beta = 1$: The density is perfectly flat (Uniform distribution over $[0, 1]$), representing equal belief across all CTRs.

*   $\alpha < \beta$ ($\alpha=2, \beta=8$): The center of mass shifts to the left (right-skewed), reflecting a belief that low conversion rates are more probable.

*   $\alpha > \beta$ ($\alpha=8, \beta=2$): The center of mass shifts to the right (left-skewed), placing higher probability mass near $1.0$.

*   The mean of the distribution is $\mathbb{E}[\Theta] = \frac{\alpha}{\alpha + \beta}$. Increasing $\alpha$ relative to $\beta$ shifts the center of mass toward $1$, while increasing $\beta$ shifts it toward $0$.

---






2. Sequential Likelihood and Joint History

For a single interaction $y_k \in \{0, 1\}$, the likelihood contribution is Bernoulli:

$$L(y_k \mid \theta) = \theta^{y_k} (1 - \theta)^{1 - y_k}$$

Assuming independent conditional responses given $\theta$, the joint likelihood function across $k$ impressions $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ is:

$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^k \theta^{y_i} (1 - \theta)^{1 - y_i} = \theta^{\sum_{i=1}^k y_i} (1 - \theta)^{k - \sum_{i=1}^k y_i}$$

Let $s_k = \sum_{i=1}^k y_i$ be the cumulative clicks observed through step $k$. The joint likelihood simplifies to:

$$L(\mathbf{y}^{(k)} \mid \theta) = \theta^{s_k} (1 - \theta)^{k - s_k}$$

---

3. Closed-Form Analytical Updates (Beta-Binomial Conjugacy)

DerivationUsing Bayes' Theorem, the posterior density at step $k$ given prior state $f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta) = \text{Beta}(\alpha_{k-1}, \beta_{k-1})$ and observation $y_k$ is:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto L(y_k \mid \theta) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$

Substitute the algebraic forms:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \left[ \theta^{y_k} (1 - \theta)^{1 - y_k} \right] \cdot \left[ \frac{1}{\mathrm{B}(\alpha_{k-1}, \beta_{k-1})} \theta^{\alpha_{k-1} - 1} (1 - \theta)^{\beta_{k-1} - 1} \right]$$

Group common base terms ($\theta$ and $1-\theta$):

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \theta^{(\alpha_{k-1} + y_k) - 1} (1 - \theta)^{(\beta_{k-1} + 1 - y_k) - 1}$$

This functional form is recognized directly as an unnormalized Beta density kernel with updated parameters:

$$\alpha_k = \alpha_{k-1} + y_k$$

$$\beta_k = \beta_{k-1} + (1 - y_k)$$

Closed-Form Parameter UpdatesAt step $k$, after $s_k$ total clicks and $k - s_k$ non-clicks:

$$\alpha_k = \alpha_0 + \sum_{i=1}^k y_i = \alpha_0 + s_k$$

$$\beta_k = \beta_0 + \sum_{i=1}^k (1 - y_i) = \beta_0 + k - s_k$$

Posterior MeanUsing the standard expected value formula for a Beta distribution:

$$\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)} = \mathbf{y}^{(k)}] = \frac{\alpha_k}{\alpha_k + \beta_k} = \frac{\alpha_0 + s_k}{\alpha_0 + \beta_0 + k}$$

---

4. Dynamic Shifting Mechanics

Analytical vs. Non-Conjugate Frameworks

1. Click ($y_k = 1$): $\alpha_k = \alpha_{k-1} + 1$ while $\beta_k = \beta_{k-1}$.
The exponent of $\theta$ increases, shifting the probability density toward higher conversion rates.

2. Non-Click ($y_k = 0$): $\alpha_k = \alpha_{k-1}$ while $\beta_k = \beta_{k-1} + 1$. The exponent of $(1-\theta)$ increases, pulling the density toward zero.


Because the Beta distribution is conjugate to the Bernoulli likelihood, the update requires zero numerical quadrature or grid integration—it is computed via simple addition in $\mathcal{O}(1)$ time.

---

5. Running Point Estimators

From the shape parameters $\alpha_k$ and $\beta_k$, the point estimates at step $k$ are:

1. Running Posterior Mean ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)$$\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \frac{\alpha_k}{\alpha_k + \beta_k}$$

2. Running Maximum A Posteriori ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

For $\alpha_k > 1$ and $\beta_k > 1$, the mode of a Beta distribution is:$$\widehat{\theta}_{\mathrm{MAP}}^{(k)} = \frac{\alpha_k - 1}{\alpha_k + \beta_k - 2}$$

---

6. Performance Tracking and Convergence Analysis

Below is the complete simulation script tracking the progression of $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$ across $n = 100$ impressions given $\theta_{\text{true}} = 0.35$.

In [2]:
import numpy as np
import plotly.graph_objects as go

# Reproducibility
np.random.seed(42)

# Parameters
n_impressions = 100
theta_true = 0.35

# 1. Base Prior Parameters (Uniform: Alpha=1, Beta=1)
alpha_0 = 1
beta_0 = 1

# 2. Simulate User Responses y_k ~ Bernoulli(theta_true)
draws = np.random.uniform(0, 1, size=n_impressions)
responses = (draws < theta_true).astype(int)

# Arrays to store estimators
steps = np.arange(0, n_impressions + 1)
bayes_estimates = np.zeros(n_impressions + 1)
map_estimates = np.zeros(n_impressions + 1)

# Step 0 initialization
alpha_k = alpha_0
beta_k = beta_0

bayes_estimates[0] = alpha_k / (alpha_k + beta_k)
# MAP under Beta(1,1) is technically undefined / uniform; set to 0.5 as midpoint
map_estimates[0] = 0.5

# 3. Running Update Loop
for k in range(1, n_impressions + 1):
    y_k = responses[k - 1]

    # Analytical Conjugate Update
    alpha_k += y_k
    beta_k += (1 - y_k)

    # Calculate Estimators
    bayes_estimates[k] = alpha_k / (alpha_k + beta_k)

    if alpha_k > 1 and beta_k > 1:
        map_estimates[k] = (alpha_k - 1) / (alpha_k + beta_k - 2)
    else:
        map_estimates[k] = bayes_estimates[k]

# 4. Plotly Line Chart
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=steps, y=bayes_estimates,
    mode='lines+markers',
    name='Posterior Mean (θ_Bayes)',
    line=dict(color='blue', width=2),
    marker=dict(size=4)
))

fig.add_trace(go.Scatter(
    x=steps, y=map_estimates,
    mode='lines+markers',
    name='MAP Estimate (θ_MAP)',
    line=dict(color='darkorange', width=2, dash='dot'),
    marker=dict(size=4)
))

fig.add_trace(go.Scatter(
    x=[0, n_impressions], y=[theta_true, theta_true],
    mode='lines',
    name='True CTR (θ_true = 0.35)',
    line=dict(color='red', width=2, dash='dash')
))

fig.update_layout(
    title="Sequential Bayesian Estimation of Ad CTR (Beta-Binomial Updates)",
    xaxis_title="Impression Step (k)",
    yaxis_title="Estimated Conversion Rate (θ)",
    template="plotly_white",
    hovermode="x unified"
)

fig.show()

Convergence Analysis

1. Shrinkage of Error: As the number of impressions $k$ approaches $100$, both $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$ settle near $\theta_{\text{true}} = 0.35$.

2. Diminishing Influence of Prior: The posterior mean formula can be rewritten as a weighted average:

$$\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \left(\frac{\alpha_0 + \beta_0}{\alpha_0 + \beta_0 + k}\right) \frac{\alpha_0}{\alpha_0 + \beta_0} + \left(\frac{k}{\alpha_0 + \beta_0 + k}\right) \bar{y}_k$$

As $k \to \infty$, the weight on the prior $\frac{\alpha_0 + \beta_0}{\alpha_0 + \beta_0 + k} \to 0$, causing the empirical sample mean $\bar{y}_k = \frac{s_k}{k}$ to dominate the estimate.

3. Variance & Certainty: The posterior variance $\operatorname{Var}(\Theta \mid \mathbf{y}^{(k)}) \approx \frac{\theta(1-\theta)}{k}$ shrinks at an $\mathcal{O}(1/k)$ rate. This variance reduction reflects the platform's increasing confidence in the advertisement's performance metric.